In [ ]:
import sys
sys.path.insert(0, '..')
from dependencies import *

In [ ]:
# --------------------------------------------
# Extract numpy arrays
# --------------------------------------------
# Replace these with your actual TRF variables
universal_encoder_log_trf = eelbrain.load.unpickle(TRF_DIR / f'universal-trf-envelope_log.pickle')
universal_encoder_onset_trf = eelbrain.load.unpickle(TRF_DIR / f'universal-trf-envelope_log_onset.pickle')


DATA_ROOT_OG = Path("~").expanduser()
# Loop over all models
for model in ['envelope', 'envelope+onset']:
    print(f'\nProcessing universal TRF for model: {model}')
    trf_list = []

    # Collect TRFs for all subjects for this model
    for subject in SUBJECTS:
        trf = eelbrain.load.unpickle(DATA_ROOT_OG / 'Data' / 'Alice' / 'TRFs' / subject / f'{subject} {model}.pickle').h_scaled
        trf_list.append(trf[0])

    # Combine and average across subjects
    alice_trf = eelbrain.combine(trf_list).mean('case')
    if model == 'envelope':
        alice_encoder_log_trf = alice_trf
    elif model == 'envelope+onset':
        alice_encoder_onset_trf = alice_trf

    print(f'Saved universal TRF for model {model}')


my_log = universal_encoder_log_trf.x
my_onset = universal_encoder_onset_trf.x

alice_log = alice_encoder_log_trf.x
alice_onset = alice_encoder_onset_trf.x


# --------------------------------------------
# Flatten TRFs
# --------------------------------------------
my_log_flat = my_log.flatten()
alice_log_flat = alice_log.flatten()

my_onset_flat = my_onset.flatten()
alice_onset_flat = alice_onset.flatten()


# --------------------------------------------
# Pearson correlation
# --------------------------------------------
r_log, p_log = pearsonr(my_log_flat, alice_log_flat)
r_onset, p_onset = pearsonr(my_onset_flat, alice_onset_flat)

print("LOG TRF correlation:", r_log, "p =", p_log)
print("ONSET TRF correlation:", r_onset, "p =", p_onset)


# --------------------------------------------
# Paired t-test
# --------------------------------------------
t_log, pt_log = ttest_rel(my_log_flat, alice_log_flat)
t_onset, pt_onset = ttest_rel(my_onset_flat, alice_onset_flat)

print("\nLOG TRF t-test: t =", t_log, "p =", pt_log)
print("ONSET TRF t-test: t =", t_onset, "p =", pt_onset)


# --------------------------------------------
# RMSE difference
# --------------------------------------------
rmse_log = np.sqrt(np.mean((my_log_flat - alice_log_flat)**2))
rmse_onset = np.sqrt(np.mean((my_onset_flat - alice_onset_flat)**2))

print("\nLOG TRF RMSE:", rmse_log)
print("ONSET TRF RMSE:", rmse_onset)


Processing universal TRF for model: envelope
Saved universal TRF for model envelope

Processing universal TRF for model: envelope+onset
Saved universal TRF for model envelope+onset
LOG TRF correlation: 0.99998564992975 p = 0.0
ONSET TRF correlation: 0.9974849610389193 p = 0.0

LOG TRF t-test: t = 0.09229441263306212 p = 0.9264667864423561
ONSET TRF t-test: t = 11.482541993941636 p = 3.0214220136454393e-30

LOG TRF RMSE: 2.476191098686519e-13
ONSET TRF RMSE: 4.364998881815652e-12
